In [ ]:
# Import everything we need
import sys, requests, json
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to path (for Jupyter notebooks)
project_root = Path.cwd().parent.parent.parent  # Go up from notebooks/ -> test/ -> src/ -> project root
sys.path.insert(0, str(project_root))

from config.constants import supabase, SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY, STORAGE_BUCKET, PDF_FILENAME
from ingest.toc_chunk import fetch_pdf_from_storage, build_toc, flatten, chunk_sections_with_hierarchy
from ingest.embedding_import import generate_embeddings, validate_embeddings
import json

# First, let's find your document
documents = supabase.table("documents") \
    .select("id, title, total_chunks, total_sections, doc_hash") \
    .execute() \
    .data

print("📚 Available documents:")
for i, doc in enumerate(documents, 1):
    print(f"\n{i}. {doc['title']}")
    print(f"   ID: {doc['id']}")
    print(f"   Hash: {doc['doc_hash']}")
    print(f"   Sections: {doc['total_sections']}")
    print(f"   Chunks: {doc['total_chunks']}")

# Set your document ID here
if len(documents) == 1:
    document_id = documents[0]['id']
    print(f"\n✅ Using document: {documents[0]['title']}")
    print(f"   Document ID: {document_id}")
else:
    print(f"\n⚠️ Multiple documents found. Set document_id manually:")
    print(f'   document_id = "{documents[0]["id"]}"')
    document_id = documents[0]['id']  # Use first one

In [ ]:
# ============================================
# CELL 11: Test Retrieval - Build Tree Function
# ============================================

def build_tree(flat_nodes):
    """Convert flat ToC nodes into nested tree structure."""
    if not flat_nodes:
        return []
    
    # Sort by page_start to maintain document order
    nodes = sorted(flat_nodes, key=lambda x: (x['page_start'], x['level']))
    
    # Add children array to each node
    for node in nodes:
        node['children'] = []
    
    # Build tree using stack
    root_nodes = []
    stack = []
    
    for node in nodes:
        # Pop nodes from stack that aren't ancestors
        while stack and stack[-1]['level'] >= node['level']:
            stack.pop()
        
        # Add to parent or root
        if stack:
            stack[-1]['children'].append(node)
        else:
            root_nodes.append(node)
        
        stack.append(node)
    
    return root_nodes


def print_tree(nodes, indent=0, max_depth=3, current_depth=0):
    """Pretty print tree structure with depth limit"""
    if current_depth >= max_depth:
        return
    
    for node in nodes:
        print("  " * indent + f"• {node['title']} (H{node['level']}, p{node['page_start']}-{node['page_end']}, DB ID: {node['id']})")
        if node['children'] and current_depth < max_depth - 1:
            print_tree(node['children'], indent + 1, max_depth, current_depth + 1)

print("✅ Tree building functions defined")

# ============================================
# CELL 12: Fetch and Build ToC Tree
# ============================================

print("=" * 80)
print("TEST: RETRIEVING AND BUILDING TOC TREE")
print("=" * 80)

# Fetch all ToC nodes for the document
toc_nodes = supabase.table("toc_nodes") \
    .select("*") \
    .eq("document_id", document_id) \
    .order("page_start") \
    .execute() \
    .data

print(f"\n✅ Retrieved {len(toc_nodes)} ToC nodes from database")

# Show level distribution
from collections import Counter
level_dist = Counter(node['level'] for node in toc_nodes)
print(f"\n📊 Distribution by level:")
for level in sorted(level_dist.keys()):
    print(f"   H{level}: {level_dist[level]} sections")

# Build tree
toc_tree = build_tree(toc_nodes)
print(f"\n✅ Built tree with {len(toc_tree)} root chapters")

# Display tree structure (first 2 chapters, max 3 levels deep)
print(f"\n📚 Document Structure (first 2 chapters, 3 levels deep):")
print("=" * 80)
print_tree(toc_tree[:2], max_depth=3)

# Save for later use
print(f"\n💾 Saved toc_tree and toc_nodes for next cells")

# ============================================
# CELL 13: Inspect Specific ToC Node
# ============================================

print("=" * 80)
print("TEST: INSPECTING SPECIFIC TOC NODE")
print("=" * 80)

# Let's examine a specific node (pick an H2 or H3 for testing)
test_candidates = [n for n in toc_nodes if n['level'] == 2]

if not test_candidates:
    test_candidates = [n for n in toc_nodes if n['level'] == 1]

test_node = test_candidates[0] if test_candidates else toc_nodes[0]

print(f"\n🎯 Selected test node:")
print(f"   DB ID: {test_node['id']}")
print(f"   node_id: {test_node['node_id']}")
print(f"   Title: {test_node['title']}")
print(f"   Level: H{test_node['level']}")
print(f"   Pages: {test_node['page_start']} - {test_node['page_end']}")

# Find this node in the tree to see its context
def find_node_path(tree, target_id, path=[]):
    """Find path to node in tree"""
    for node in tree:
        current_path = path + [node['title']]
        if node['id'] == target_id:
            return current_path
        if node['children']:
            result = find_node_path(node['children'], target_id, current_path)
            if result:
                return result
    return None

path = find_node_path(toc_tree, test_node['id'])
if path:
    print(f"\n📍 Location in document:")
    print("   " + " → ".join(path))

# Check how many chunks this section has
chunk_count = supabase.table("chunks") \
    .select("id", count="exact") \
    .eq("document_id", document_id) \
    .eq("toc_node_id", test_node['id']) \
    .execute()

print(f"\n📦 This section has {chunk_count.count} chunks")

In [ ]:
# ============================================
# CELL 14: Retrieve Section Content
# ============================================

print("\n" + "=" * 80)
print("TEST: RETRIEVING SECTION CONTENT")
print("=" * 80)

# Function to get section content
def get_section_content(document_id, toc_node_db_id):
    """Get all chunks for a specific ToC node"""
    chunks = supabase.table("chunks") \
        .select("id, chunk_id, chunk_seq, section_title, text, page_start, page_end, level") \
        .eq("document_id", document_id) \
        .eq("toc_node_id", toc_node_db_id) \
        .order("chunk_seq") \
        .execute() \
        .data
    
    return chunks

# Get chunks for our test node
section_chunks = get_section_content(document_id, test_node['id'])

print(f"✅ Retrieved {len(section_chunks)} chunks for section: '{test_node['title']}'")

if section_chunks:
    # Combine all chunk text
    full_text = "\n\n".join(c['text'] for c in section_chunks)
    total_words = len(full_text.split())
    
    print(f"\n📊 Section statistics:")
    print(f"   Total characters: {len(full_text):,}")
    print(f"   Total words: {total_words:,}")
    print(f"   Number of chunks: {len(section_chunks)}")
    print(f"   Avg words per chunk: {total_words / len(section_chunks):.0f}")
    
    print(f"\n📝 Chunk details:")
    for i, chunk in enumerate(section_chunks[:3]):  # Show first 3
        print(f"\n   Chunk {i+1}:")
        print(f"      chunk_seq: {chunk['chunk_seq']}")
        print(f"      pages: {chunk['page_start']}-{chunk['page_end']}")
        print(f"      length: {len(chunk['text'])} chars, {len(chunk['text'].split())} words")
        print(f"      preview: {chunk['text'][:150]}...")
    
    if len(section_chunks) > 3:
        print(f"\n   ... and {len(section_chunks) - 3} more chunks")
    
    print(f"\n📖 Full section content (first 800 characters):")
    print("=" * 80)
    print(full_text[:800])
    print("...")
    print("=" * 80)
else:
    print("⚠️ No chunks found for this section")

In [ ]:
# Test the search function
print("="*70)
print("TESTING SEARCH")
print("="*70)

# Use the document_id from Cell 3
query = "What is prompt engineering?"
print(f"\nQuery: '{query}'")
print("-"*70)

results = search(query, document_id=document_id, top_k=5)

if results:
    print(f"\nFound {len(results)} results:\n")
    
    for i, r in enumerate(results, 1):
        print(f"{i}. [{r['similarity']:.3f}] {r['section_title']}")
        print(f"   Level {r['level']} | Pages {r['page_start']}-{r['page_end']}")
        print(f"   {r['text'][:150]}...")
        print()
else:
    print("No results found")

In [ ]:
# ============================================
# CELL 15: Test Multi-Section Retrieval (with children)
# ============================================

print("\n" + "=" * 80)
print("TEST: RETRIEVING SECTION WITH CHILDREN")
print("=" * 80)

# Pick an H1 chapter that has children
test_chapter = None
for node in [n for n in toc_nodes if n['level'] == 1]:
    # Count children
    children_count = len([n for n in toc_nodes 
                          if n['page_start'] >= node['page_start'] 
                          and n['page_end'] <= node['page_end']
                          and n['level'] > node['level']])
    if children_count > 0:
        test_chapter = node
        break

if not test_chapter:
    test_chapter = [n for n in toc_nodes if n['level'] == 1][0]

print(f"🎯 Testing with chapter: '{test_chapter['title']}'")
print(f"   Pages: {test_chapter['page_start']}-{test_chapter['page_end']}")

# Get all descendant nodes (subsections)
descendant_nodes = [n for n in toc_nodes 
                    if n['page_start'] >= test_chapter['page_start']
                    and n['page_end'] <= test_chapter['page_end']
                    and n['id'] >= test_chapter['id']]  # Include self

print(f"   Total subsections (including self): {len(descendant_nodes)}")

# Get all ToC node DB IDs
node_ids = [n['id'] for n in descendant_nodes]

# Fetch all chunks for these nodes
all_chunks = supabase.table("chunks") \
    .select("id, chunk_seq, section_title, text, page_start, level, toc_node_id") \
    .eq("document_id", document_id) \
    .in_("toc_node_id", node_ids) \
    .order("page_start, chunk_seq") \
    .execute() \
    .data

print(f"\n✅ Retrieved {len(all_chunks)} chunks across all subsections")

# Show distribution by section
from collections import defaultdict
chunks_by_section = defaultdict(list)
for chunk in all_chunks:
    chunks_by_section[chunk['toc_node_id']].append(chunk)

print(f"\n📊 Chunks per subsection:")
for node in descendant_nodes[:5]:  # Show first 5
    count = len(chunks_by_section[node['id']])
    print(f"   {'  ' * (node['level'] - 1)}{node['title']}: {count} chunks")

# Combine full text
full_chapter_text = "\n\n".join(c['text'] for c in all_chunks)
print(f"\n📖 Full chapter statistics:")
print(f"   Total characters: {len(full_chapter_text):,}")
print(f"   Total words: {len(full_chapter_text.split()):,}")
print(f"   Total chunks: {len(all_chunks)}")

In [ ]:
# ============================================
# CELL 16: Test User Selection Workflow
# ============================================

print("\n" + "=" * 80)
print("TEST: SIMULATING USER SELECTION WORKFLOW")
print("=" * 80)

def get_section_for_assessment(document_id, node_id_string, include_children=True):
    """
    Simulate what happens when user selects a section.
    
    Args:
        document_id: UUID of document
        node_id_string: The node_id string (e.g., "h2-1-1__section-title")
        include_children: Whether to include subsections
    """
    # 1. Get the ToC node by node_id
    node_result = supabase.table("toc_nodes") \
        .select("*") \
        .eq("document_id", document_id) \
        .eq("node_id", node_id_string) \
        .single() \
        .execute()
    
    node = node_result.data
    
    print(f"✅ Found section: '{node['title']}'")
    print(f"   DB ID: {node['id']}")
    print(f"   Level: H{node['level']}")
    print(f"   Pages: {node['page_start']}-{node['page_end']}")
    
    # 2. Determine which nodes to include
    if include_children:
        # Get all descendants
        all_nodes_result = supabase.table("toc_nodes") \
            .select("id") \
            .eq("document_id", document_id) \
            .gte("page_start", node["page_start"]) \
            .lte("page_end", node["page_end"]) \
            .gte("level", node["level"]) \
            .execute()
        
        node_ids = [n["id"] for n in all_nodes_result.data]
        print(f"   Including {len(node_ids)} sections (with children)")
    else:
        node_ids = [node["id"]]
        print(f"   Single section only")
    
    # 3. Get chunks
    chunks = supabase.table("chunks") \
        .select("id, chunk_seq, section_title, text, page_start") \
        .eq("document_id", document_id) \
        .in_("toc_node_id", node_ids) \
        .order("page_start, chunk_seq") \
        .execute() \
        .data
    
    # 4. Combine text
    full_text = "\n\n".join(c["text"] for c in chunks)
    
    return {
        "section": node,
        "text": full_text,
        "chunks": chunks,
        "total_words": len(full_text.split()),
        "total_chars": len(full_text)
    }

# Test with our test node
print("\n" + "=" * 80)
print("Scenario 1: User selects section WITHOUT children")
print("=" * 80)

result1 = get_section_for_assessment(
    document_id, 
    test_node['node_id'], 
    include_children=False
)

print(f"\n📊 Assessment context:")
print(f"   Section: {result1['section']['title']}")
print(f"   Total words: {result1['total_words']:,}")
print(f"   Total chunks: {len(result1['chunks'])}")
print(f"\n📖 Content preview (first 500 chars):")
print(result1['text'][:500] + "...")

print("\n" + "=" * 80)
print("Scenario 2: User selects section WITH children")
print("=" * 80)

# Use a chapter node
chapter_node = [n for n in toc_nodes if n['level'] == 1][0]

result2 = get_section_for_assessment(
    document_id,
    chapter_node['node_id'],
    include_children=True
)

print(f"\n📊 Assessment context:")
print(f"   Section: {result2['section']['title']}")
print(f"   Total words: {result2['total_words']:,}")
print(f"   Total chunks: {len(result2['chunks'])}")

In [ ]:
# ============================================
# CELL 17: Test Semantic Search (Optional) - FIXED
# ============================================

print("\n" + "=" * 80)
print("TEST: SEMANTIC SEARCH WITHIN SELECTED SECTION")
print("=" * 80)

from ingest.embedding_import import generate_embeddings
import json

# Scenario: User asks a follow-up question
test_query = "What are the main concepts discussed?"
print(f"🔍 Query: '{test_query}'")

# Generate query embedding
query_embedding = generate_embeddings(
    [test_query],
    model_name="BAAI/bge-small-en-v1.5",
    show_progress=False
)[0]

print(f"✅ Generated query embedding: {query_embedding.shape}")

# Get chunks from our test section
section_chunks_with_embeddings = supabase.table("chunks") \
    .select("id, chunk_id, section_title, text, embedding") \
    .eq("document_id", document_id) \
    .eq("toc_node_id", test_node['id']) \
    .execute() \
    .data

if section_chunks_with_embeddings:
    print(f"🔎 Searching {len(section_chunks_with_embeddings)} chunks in section...")
    
    # Compute similarities
    similarities = []
    for chunk in section_chunks_with_embeddings:
        # FIX: Convert embedding from string/list to numpy array
        chunk_embedding_raw = chunk['embedding']
        
        # Handle different formats
        if isinstance(chunk_embedding_raw, str):
            # If it's a string, parse as JSON
            chunk_embedding = np.array(json.loads(chunk_embedding_raw))
        elif isinstance(chunk_embedding_raw, list):
            # If it's already a list, convert to numpy
            chunk_embedding = np.array(chunk_embedding_raw)
        else:
            # Already numpy array
            chunk_embedding = chunk_embedding_raw
        
        # Ensure float type
        chunk_embedding = chunk_embedding.astype(np.float32)
        
        # Compute cosine similarity (dot product since vectors are normalized)
        similarity = np.dot(query_embedding, chunk_embedding)
        
        similarities.append({
            'chunk_id': chunk['chunk_id'],
            'section_title': chunk['section_title'],
            'text': chunk['text'],
            'similarity': float(similarity)
        })
    
    # Sort by similarity
    similarities.sort(key=lambda x: x['similarity'], reverse=True)
    
    print(f"\n🏆 Top 3 most relevant chunks:")
    for i, result in enumerate(similarities[:3], 1):
        print(f"\n{i}. Similarity: {result['similarity']:.4f}")
        print(f"   Chunk ID: {result['chunk_id']}")
        print(f"   Preview: {result['text'][:200]}...")
    
    # Show similarity distribution
    all_sims = [s['similarity'] for s in similarities]
    print(f"\n📊 Similarity distribution:")
    print(f"   Min: {min(all_sims):.4f}")
    print(f"   Max: {max(all_sims):.4f}")
    print(f"   Mean: {np.mean(all_sims):.4f}")
    print(f"   Median: {np.median(all_sims):.4f}")
else:
    print("⚠️ No chunks found with embeddings")

In [ ]:
# ============================================
# CELL 18: Create Assessment Prompt
# ============================================

print("\n" + "=" * 80)
print("TEST: GENERATING LLM ASSESSMENT PROMPT")
print("=" * 80)

# Use the content from earlier
assessment_context = get_section_for_assessment(
    document_id,
    test_node['node_id'],
    include_children=False
)

# Build prompt for question generation
question_generation_prompt = f"""You are generating assessment questions for a student studying from their textbook.

**Section:** {assessment_context['section']['title']}
**Level:** H{assessment_context['section']['level']}
**Pages:** {assessment_context['section']['page_start']}-{assessment_context['section']['page_end']}

**Content:**
{assessment_context['text']}

---

Generate 5 questions that test understanding of this specific section:
1. Mix of factual recall and conceptual understanding
2. Reference specific concepts from the text
3. Indicate the page number where the answer can be found
4. Make questions challenging but fair

Format as:
Q1: [question]
Answer location: Page X
Difficulty: [Easy/Medium/Hard]
"""

print("📝 Generated prompt for LLM:")
print("=" * 80)
print(question_generation_prompt[:1500])
print("\n... [truncated for display]")
print("=" * 80)

print(f"\n📊 Prompt statistics:")
print(f"   Total characters: {len(question_generation_prompt):,}")
print(f"   Total words: {len(question_generation_prompt.split()):,}")
print(f"   Estimated tokens: ~{len(question_generation_prompt) / 4:.0f}")